# Styling Pandas DataFrames

Raw DataFrames are functional but hard to read. Pandas provides two complementary systems for making them **visually communicative**:

| System | Purpose | What it changes |
|---|---|---|
| **Options/Settings API** | Control display limits | Rows shown, column widths, decimal precision |
| **Styling API** | Apply CSS visual formatting | Colors, fonts, bars, conditional highlights |

**Key concept:** The Styling API works by attaching CSS to the rendered HTML table output. It uses the same CSS `key: value` pair syntax you'd use in a stylesheet.

**Important note:** Style formatting returns a **Styler object**, not a DataFrame. The styled output is only visible in Jupyter notebooks or other HTML-rendering environments.

---

# Import Packages

In [ ]:
# pandas: the primary data manipulation library — provides DataFrame and Styler
import pandas as pd

# numpy: numerical computing library — used for generating sample data and computations
import numpy as np

# Import Data

In [ ]:
# Load the salaries dataset from a CSV file into a DataFrame
# filepath_or_buffer: the path to the CSV file (relative to the notebook location)
# df is our main DataFrame — all styling examples will use this
df = pd.read_csv(filepath_or_buffer='../Data/salaries.csv')

# Preview the first few rows to understand the data structure
# Expected columns: EmployeeName, JobTitle, BasePay, OvertimePay, OtherPay,
#                   Benefits, TotalPay, TotalPayBenefits, Year, Status
df.head()

---

# Pandas Options / Settings API

The Options API controls **how pandas displays data** in the output — not what the data contains.

**Key functions:**

| Function | Purpose |
|---|---|
| `pd.set_option(key, value)` | Set a display option |
| `pd.get_option(key)` | Get the current value of an option |
| `pd.reset_option(key)` | Reset an option back to its default |
| `pd.describe_option()` | List all available options with descriptions |

The most common problem this solves: DataFrames with many rows/columns truncate to `...` in the middle. These options let you control exactly how much is shown.

## Controlling How Many Rows and Columns Are Displayed

In [ ]:
# Limit the maximum number of rows shown in any DataFrame display
# When a DataFrame has more rows than this, the middle rows are replaced with '...'
# Example: a 1000-row DataFrame will show 3 rows at top, '...', 3 rows at bottom (total ~7)
pd.set_option('display.max_rows', 7)

# Limit the maximum number of columns shown in any DataFrame display
# Columns beyond this limit are truncated with '...' in the middle
# Useful for wide DataFrames with many columns
pd.set_option('display.max_columns', 7)

# Verify the settings were applied
print(f"max_rows is now: {pd.get_option('display.max_rows')}")
print(f"max_columns is now: {pd.get_option('display.max_columns')}")

## Controlling Column Width, Precision, and Threshold

In [ ]:
# Set the maximum number of characters shown in a single column cell
# Longer strings will be truncated with '...' at character 40
# Prevents very long strings (URLs, descriptions) from widening the table
pd.set_option('max_colwidth', 40)

# Set the number of decimal places shown for floating-point numbers
# Only affects DISPLAY — the underlying float value is NOT rounded
# Example: 3.141592653... → displayed as 3.1416
pd.set_option('display.precision', 4)

# Values below this threshold are displayed as zero (for cleaner output)
# Helps declutter tables with many near-zero values
# Example: 0.3 would display as 0 (since 0.3 < 0.5); 0.6 would display normally
pd.set_option('chop_threshold', .5)

print("Column width, precision, and threshold options set.")

## Exploring All Available Options

In [ ]:
# Display ALL available pandas display options with their descriptions and defaults
# This produces a lot of output — useful as a reference for discovering options
# You can search for specific options by passing a pattern: pd.describe_option('display')
pd.describe_option()  # Scroll through output to see all configurable options

---

# Pandas Styling API

The Styling API transforms DataFrames into **visually rich HTML tables** by applying CSS.

You write **style functions** that receive data values and return CSS strings, which are then applied to the rendered cells.

**How it works:**
1. Call `.style` on a DataFrame → returns a `Styler` object
2. Chain styling methods on the `Styler` (`.format()`, `.highlight_max()`, `.map()`, etc.)
3. Jupyter renders the Styler as a styled HTML table

---

## Formatting Currency Values with `.format()`

The most common use case: displaying numeric columns as currency with dollar signs and comma separators.

`style.format()` accepts a dictionary mapping **column names → format strings**.

In [ ]:
# Apply currency formatting directly in the method call
# style.format() accepts a dict: {column_name: format_string}

# Format string breakdown for "${:20,.0f}":
#   $        → literal dollar sign prepended to the output
#   {:20,.0f}  → Python format spec:
#      20    → minimum width of 20 characters (right-aligned by default)
#      ,     → use comma as thousands separator (e.g., 14,900)
#      .0    → zero decimal places (round to nearest integer)
#      f     → format as fixed-point float
df.head(10).style.format({
    "BasePay":         "${:20,.0f}",   # e.g., $      14,900
    "OvertimePay":     "${:,.0f}",     # e.g., $2,000
    "OtherPay":        "${:,.0f}",     # e.g., $500
    "TotalPay":        "${:20,.0f}",   # e.g., $      17,400
    "TotalPayBenefits":"${:20,.0f}"    # e.g., $      20,000
})

## Storing Format Specs in a Dictionary Variable

Inline format dictionaries are hard to reuse and maintain. Store them in named variables instead.

In [ ]:
# Store all currency column format strings in a dedicated dictionary variable
# This makes the format spec reusable and easy to maintain in one place
ccy_formats = {
    "BasePay":          "${:20,.0f}",   # Base salary formatted as currency
    "OvertimePay":      "${:,.0f}",     # Overtime pay formatted as currency
    "OtherPay":         "${:,.0f}",     # Other pay formatted as currency
    "TotalPay":         "${:20,.0f}",   # Total pay formatted as currency
    "TotalPayBenefits": "${:20,.0f}"    # Total pay + benefits formatted as currency
}

# Pass the dictionary using the 'formatter' keyword argument
# This is equivalent to the inline version above but much cleaner
df.head(10).style.format(formatter=ccy_formats)

## The Problem with Chaining Multiple `.format()` Calls

You might assume you can chain `.format()` calls to apply different formats to different columns. This doesn't work as expected — each new `.format()` call **replaces** the previous one.

In [ ]:
# Define a second format dictionary for text columns
# lambda x: x.lower() converts text to lowercase
name_formats = {
    "JobTitle":      lambda x: x.lower(),       # e.g., 'MANAGER' → 'manager'
    "EmployeeName":  lambda x: x.lower()        # e.g., 'ALICE SMITH' → 'alice smith'
}

# ⚠️  PROBLEM: Chaining two .format() calls does NOT combine them!
# style.format() returns a NEW Styler — the second .format() call replaces the first
# Result: only name_formats is applied; ccy_formats is LOST
# This is a known gotcha in the pandas Styling API
df.head(10).style.format(formatter=ccy_formats).format(formatter=name_formats)

# Expected: both currency AND name formatting applied
# Actual: only name formatting applied (currency formatting silently discarded)

---

## Incremental Styling Workaround — Merge Formatter Dictionaries

The fix: **merge all format dictionaries into one** before calling `.style.format()`. Only one `.format()` call is needed.

In [ ]:
# Helper function to merge two formatter dictionaries into one combined dictionary
# d1: first format dictionary (e.g., currency formats)
# d2: second format dictionary (e.g., text formats)
def merge_formatters(d1, d2):
    # Create a new empty dictionary to hold the combined result
    d3 = dict()

    # Copy all key-value pairs from d1 into d3
    d3.update(d1)

    # Copy all key-value pairs from d2 into d3
    # If a key exists in both d1 and d2, d2's value wins (d2 takes precedence)
    d3.update(d2)

    # Return the unified dictionary containing all formatters from both inputs
    return d3

In [ ]:
# Merge both format dictionaries into a single combined formatter
# col_formats now contains ALL column format specs from ccy_formats AND name_formats
col_formats = merge_formatters(ccy_formats, name_formats)

# Apply ALL formatting in a single .format() call — no more chaining problem!
# Currency columns get dollar signs; name columns get lowercased
df.head(10).style.format(formatter=col_formats)

## Hiding the Index

By default, every DataFrame displays a numeric index column (0, 1, 2, ...) on the left. For presentation purposes, this index is often irrelevant and visually clutters the table.

In [ ]:
# .hide() without arguments hides the row index (the 0, 1, 2, ... column on the left)
# This is purely cosmetic — the underlying DataFrame still has its index
# Chained after .format() so both formatting AND index hiding apply together
df.head(10).style.format(formatter=col_formats).hide()

# You can also hide specific columns:
# .hide(['BasePay', 'OvertimePay'], axis='columns')  → hides those columns

---

# Highlighters — Conditional Visual Formatting

Highlighters apply CSS background colors to cells based on their values. Pandas provides built-in highlighters for common cases, and you can write custom ones too.

This is equivalent to **conditional formatting** in Excel or Google Sheets.

## Built-in Highlighters: Max, Min, and Gradient

In [ ]:
# highlight_max(): applies a background color to the cell with the HIGHEST value
#   color='lightgreen' → the maximum cell gets a light green background
# highlight_min(): applies a background color to the cell with the LOWEST value
#   color='red' → the minimum cell gets a red background
# Both methods work column-by-column by default (each column has its own max/min)
(
    df.head(10)
    .style
    .format(formatter=col_formats)        # Apply number and text formatting
    .hide()                               # Hide the row index
    .highlight_max(color='lightgreen')    # Green background on highest value per column
    .highlight_min(color='red')           # Red background on lowest value per column
)

In [ ]:
# background_gradient(): applies a color gradient across an entire column based on value
# Cells with higher values get deeper/darker color; lower values get lighter color
#   cmap='Blues' → uses matplotlib's 'Blues' colormap (light blue to dark blue)
# Other useful cmaps: 'Greens', 'Reds', 'RdYlGn' (red-yellow-green), 'coolwarm'
(
    df.head(10)
    .style
    .format(formatter=col_formats)     # Apply all column formats
    .hide()                            # Hide row index
    .background_gradient(cmap='Blues') # Gradient coloring: lighter = lower, darker = higher
)

---

# Styler Properties — Global Cell Styling

`set_properties()` applies a **fixed CSS style** to ALL cells in the table (or a subset you specify).

It doesn't respond to data values — it just sets uniform visual properties like background color, text color, border style, font size, etc.

In [ ]:
# Define a dictionary of CSS properties to apply uniformly to all cells
# Keys are CSS property names; values are CSS property values
styler_properties = {
    'background-color': 'black',   # Black background — 'Matrix' style dark theme
    'color': 'lawngreen',          # Bright green text (CSS named color)
    'border-color': 'white'        # White borders between cells for visibility
}

# ** unpacks the dictionary as keyword arguments to set_properties()
# Equivalent to: set_properties(**{'background-color': 'black', ...})
# This applies the CSS properties to EVERY cell in the first 10 rows
df.head(10).style.set_properties(**styler_properties)

---

# `Styler.map()` — Element-Wise Custom Styling

For full control, write your own **style function** and apply it with `.map()` or `.apply()`.

| Method | Applies function | Input to function | Typical use |
|---|---|---|---|
| `.map(func)` | Element-by-element | Single scalar value | Style based on a cell's own value |
| `.apply(func, axis=0)` | Column-by-column | Entire column (Series) | Style based on comparison within a column |
| `.apply(func, axis=1)` | Row-by-row | Entire row (Series) | Style based on comparison within a row |
| `.apply(func, axis=None)` | Whole table | Entire DataFrame | Style based on comparisons across the whole table |

Style functions must return a **CSS string** like `'color: red'` or `'background-color: yellow'`.

In [ ]:
# Custom style function: color text RED for strings, BLACK for numbers
# This function is called once per cell — 'x' is the individual cell value
def red_or_black(x):
    # isinstance(x, str) → True if the cell contains a string value
    # If string → return CSS for red text; otherwise → return CSS for black text
    # f-string builds the CSS property string dynamically based on the condition
    return f"color:{'red' if isinstance(x, str) else 'black'}"

# .map(func) applies red_or_black() to EVERY cell in the DataFrame
# Each cell gets either 'color:red' or 'color:black' depending on its type
df.head(10).style.format(formatter=col_formats).hide().map(red_or_black)

---

# Bar Charts Within DataFrames

The Styling API can render **mini horizontal bar charts** directly inside DataFrame cells — proportional to each cell's value relative to the column's range.

This is an extremely powerful visualization technique: it makes magnitude differences immediately visible without needing a separate chart.

In [ ]:
# .bar() renders horizontal bar charts inside cells, proportional to each value
# subset=[col_name]: list of column names to apply the bar chart to
# color: the color of the bar fill (any valid CSS color — named, hex, or rgb)

# Start by sorting by TotalPay descending so highest earners appear first
(
    df.head(10)
    .sort_values(by='TotalPay', ascending=False)  # Highest earners at top
    .style
    .format(formatter=col_formats)                 # Apply all column formats
    .hide()                                        # Hide the row index

    # Bar chart for OtherPay column — light green bars
    # Bar width is proportional to each row's OtherPay relative to max OtherPay
    .bar(subset=["OtherPay"], color='lightgreen')

    # Bar chart for BasePay column — hot pink bars
    # Each bar shows what fraction of max BasePay this employee earns
    .bar(subset=["BasePay"], color='#ee1f5f')

    # Bar chart for TotalPay column — salmon/orange bars
    # The longest bar = highest TotalPay; others are proportionally shorter
    .bar(subset=["TotalPay"], color='#FFA07A')
)

---

## Summary — Pandas Styling Cheat Sheet

```python
import pandas as pd

# --- OPTIONS API ---
pd.set_option('display.max_rows', 10)        # Show max 10 rows
pd.set_option('display.max_columns', 8)      # Show max 8 columns
pd.set_option('display.precision', 2)        # 2 decimal places
pd.set_option('max_colwidth', 30)            # Truncate cells at 30 chars
pd.reset_option('display.max_rows')          # Reset to default

# --- STYLING API (returns Styler, not DataFrame) ---

# Format specific columns
df.style.format({'Price': '${:,.2f}', 'Name': str.upper})

# Hide index
df.style.hide()

# Built-in highlighters
df.style.highlight_max(color='lightgreen')
df.style.highlight_min(color='tomato')
df.style.background_gradient(cmap='Blues')

# Global cell properties
df.style.set_properties(**{'background-color': 'black', 'color': 'white'})

# Custom element-wise style function
df.style.map(lambda x: 'color: red' if x < 0 else 'color: black')

# Bar charts inside cells
df.style.bar(subset=['Value'], color='steelblue')

# Chain multiple styles (use merge_formatters to combine format dicts)
df.style.format(formatter=col_formats).hide().highlight_max(color='lime')
```

> **Remember:** `.style` returns a `Styler` object. Styled DataFrames only render visually in Jupyter notebooks. Calling `print()` or working with them programmatically behaves like a regular DataFrame.